# Day 2: Data Cleaning & SQL Database Design
**Project:** Bluestock Mutual Fund Analytics Platform  
**Notebook:** 02_data_cleaning.ipynb  
**Objectives:**
1. Clean & validate all datasets (dates, duplicate handling, missing value imputation, type casting).
2. Compute derived daily returns for the NAV time series.
3. Design and implement a 5+ table relational SQLite database (`bluestock_mf.db`).
4. Execute and inspect 10 analytical SQL queries.


In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

DB_PATH = os.path.join("..", "data", "db", "bluestock_mf.db")
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.join("data", "db", "bluestock_mf.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "bluestock_mf.db"

conn = sqlite3.connect(DB_PATH)
print("Connected to SQLite Database:", DB_PATH)


## 1. Verify Database Schema & Table Row Counts

In [ ]:
cursor = conn.cursor()
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';").fetchall()
table_names = [t[0] for t in tables]

print(f"Total tables in SQLite database: {len(table_names)}\n")
for tbl in table_names:
    count = cursor.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"Table: {tbl:<25} | Row Count: {count:,}")


## 2. Execute 10 Analytical SQL Queries

### Query 1: Top 5 Mutual Funds by AUM

In [ ]:
q1 = """
SELECT 
    f.amfi_code,
    f.scheme_name,
    f.fund_house,
    f.category,
    p.aum_crore,
    p.morningstar_rating
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
"""
pd.read_sql(q1, conn)


### Query 2: Monthly Average NAV for Key Scheme (SBI Bluechip 119551)

In [ ]:
q2 = """
SELECT 
    f.scheme_name,
    SUBSTR(n.date, 1, 7) AS year_month,
    ROUND(AVG(n.nav), 2) AS avg_nav,
    ROUND(MIN(n.nav), 2) AS min_nav,
    ROUND(MAX(n.nav), 2) AS max_nav
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
WHERE n.amfi_code = 119551
GROUP BY f.scheme_name, SUBSTR(n.date, 1, 7)
ORDER BY year_month DESC
LIMIT 12;
"""
pd.read_sql(q2, conn)


### Query 3: Monthly SIP Inflow & YoY Growth

In [ ]:
q3 = """
SELECT 
    month,
    sip_inflow_crore,
    active_sip_accounts_crore,
    ROUND(yoy_growth_pct, 2) AS yoy_growth_pct,
    sip_aum_lakh_crore
FROM fact_sip_inflows
ORDER BY month DESC
LIMIT 12;
"""
pd.read_sql(q3, conn)


### Query 4: Transaction Volume and Amount by State

In [ ]:
q4 = """
SELECT 
    state,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr) / 10000000.0, 2) AS total_invested_crore,
    ROUND(AVG(amount_inr), 2) AS avg_transaction_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_invested_crore DESC
LIMIT 10;
"""
pd.read_sql(q4, conn)


### Query 5: Funds with Expense Ratio < 1.0%

In [ ]:
q5 = """
SELECT 
    f.amfi_code,
    f.scheme_name,
    f.fund_house,
    f.sub_category,
    f.plan,
    f.expense_ratio_pct,
    p.return_3yr_pct,
    p.sharpe_ratio
FROM dim_fund f
JOIN fact_performance p ON f.amfi_code = p.amfi_code
WHERE f.expense_ratio_pct < 1.0
ORDER BY f.expense_ratio_pct ASC;
"""
pd.read_sql(q5, conn)


### Query 6: Inflow Distribution Across Categories

In [ ]:
q6 = """
SELECT 
    category,
    ROUND(SUM(net_inflow_crore), 2) AS total_net_inflow_cr
FROM fact_category_inflows
GROUP BY category
ORDER BY total_net_inflow_cr DESC;
"""
pd.read_sql(q6, conn)


### Query 7: Fund House AUM Market Share

In [ ]:
q7 = """
SELECT 
    fund_house,
    aum_crore,
    ROUND(aum_crore * 100.0 / (SELECT SUM(aum_crore) FROM fact_aum WHERE date = '2024-03-31'), 2) AS market_share_pct
FROM fact_aum
WHERE date = '2024-03-31'
ORDER BY aum_crore DESC;
"""
pd.read_sql(q7, conn)


### Query 8: Top 10 Heavily Weighted Stock Holdings Across Portfolios

In [ ]:
q8 = """
SELECT 
    stock_symbol,
    stock_name,
    sector,
    COUNT(DISTINCT amfi_code) AS held_by_num_funds,
    ROUND(AVG(weight_pct), 2) AS avg_portfolio_weight_pct,
    ROUND(SUM(market_value_cr), 2) AS total_market_value_cr
FROM fact_portfolio_holdings
GROUP BY stock_symbol, stock_name, sector
ORDER BY total_market_value_cr DESC
LIMIT 10;
"""
pd.read_sql(q8, conn)


### Query 9: Transaction Breakdown by City Tier (T30 vs B30)

In [ ]:
q9 = """
SELECT 
    city_tier,
    transaction_type,
    COUNT(*) AS txn_count,
    ROUND(SUM(amount_inr) / 10000000.0, 2) AS total_amount_cr,
    ROUND(AVG(amount_inr), 2) AS avg_ticket_size
FROM fact_transactions
GROUP BY city_tier, transaction_type
ORDER BY city_tier, total_amount_cr DESC;
"""
pd.read_sql(q9, conn)


### Query 10: Risk-Adjusted Return Leaders (Sharpe Ratio)

In [ ]:
q10 = """
SELECT 
    f.scheme_name,
    f.sub_category,
    p.return_3yr_pct,
    p.sharpe_ratio,
    p.sortino_ratio,
    p.alpha,
    p.beta,
    p.risk_grade
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.sharpe_ratio DESC
LIMIT 10;
"""
pd.read_sql(q10, conn)
